# Phase 2 – Regularized Logistic Regression (biopsy / no biopsy)

**Input:** the files produced by Phase 1 (`features_clean.csv`, `embeddings_aligned.csv`, `feature_lists.json`).

**Roadmap**
1. Load data and keep only the labelled patients
2. Set up the model as one **Pipeline** (impute → scale → logistic regression)
3. Evaluate with **repeated stratified k-fold CV** and compare with **simple baselines**
4. **Tune** the regularization strength using **nested CV** (honest score)
5. *(Experiment)* Do the MRI embeddings help?
6. Fit the final model, **read the coefficients**
7. Look at errors, threshold, calibration and the clinician-confidence file
8. Predict the 104 unlabelled patients

Each block starts with **Why** so you know the reasoning behind the code.

## 1. Load data

**Why:** supervised learning needs a known answer (`label`). The 104 patients without a label cannot be used to train or to score the model, so we separate them now and keep them only for the final predictions.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, cross_validate)
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
warnings.filterwarnings('ignore')
pd.set_option('display.width', 150)

DATA_DIR = '.'
data = pd.read_csv(f'{DATA_DIR}/features_clean.csv')
emb  = pd.read_csv(f'{DATA_DIR}/embeddings_aligned.csv')       # same row order as `data`
meta = json.load(open(f'{DATA_DIR}/feature_lists.json'))

# Very rare flags (<5 positives among labelled patients) cannot be learned from; drop them.
# (This uses only how often a flag occurs, never how it relates to the label.)
RARE = ['flag_anticoagulant', 'dre_not_done', 'zone_unknown']
FEATURES = [c for c in meta['features'] if c not in RARE]

is_labelled = data['label'].notna()
X_all, ids_all = data[FEATURES], data['case_id']
X, y = X_all[is_labelled], data.loc[is_labelled, 'label'].astype(int)
ids = ids_all[is_labelled]

print(f'labelled patients: {len(y)}  | unlabelled: {(~is_labelled).sum()}  | features: {len(FEATURES)}')
print('biopsy (label=1): %d (%.0f%%)   no biopsy (label=0): %d' % (y.sum(), 100*y.mean(), (1-y).sum()))

labelled patients: 91  | unlabelled: 104  | features: 56
biopsy (label=1): 56 (62%)   no biopsy (label=0): 35


## 2. The model as one Pipeline

**Why a Pipeline?** Imputation and scaling *learn numbers from the data* (e.g. the median PSA). If we learned them from all patients first, the test patients would secretly influence training (data leakage). Inside a Pipeline, scikit-learn re-learns them **only on the training part of every CV split**.

| Step | What it does | Why |
|---|---|---|
| `SimpleImputer(median)` | fills remaining missing values with the training median | the model cannot handle blanks; median is not distorted by outliers |
| `StandardScaler` | rescales each feature to mean 0, spread 1 | the penalty treats all coefficients equally, so features must be on the same scale; also makes coefficients comparable |
| `LogisticRegression(penalty L2)` | learns a weight per feature; **C controls regularization** (small C = strong penalty = simpler model) | with only 91 patients, a penalty stops the model memorising noise |

`class_weight=None` on purpose: the classes are only mildly unbalanced (62/38) and we want honest probabilities (e.g. "0.72 chance"), which `class_weight='balanced'` would distort.

In [2]:
def make_model(C=0.1):
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
        ('model',  LogisticRegression(C=C, penalty='l2', max_iter=5000)),
    ])

## 3. Honest evaluation: repeated stratified 5-fold CV

**Why:** with 91 patients one train/test split is too lucky/unlucky. 5-fold CV tests every patient once; *repeating* it 10 times with different shuffles gives 50 scores, so we also see how **uncertain** the result is. *Stratified* keeps the 62/38 ratio in every fold.

**Metric = AUC** (area under the ROC curve): the chance that a random biopsy patient gets a higher risk than a random no-biopsy patient. 0.5 = coin flip, 1.0 = perfect. It does not depend on a threshold, which suits a small dataset.

In [3]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

def cv_auc(model, X_, y_, name):
    scores = cross_validate(model, X_, y_, cv=cv, scoring='roc_auc')['test_score']
    print(f'{name:40s} AUC = {scores.mean():.3f}  (± {scores.std():.3f})')
    return scores

# --- Baselines: if our model cannot beat these, it adds no value ---------------------------
print('BASELINES (one feature at a time)')
for col in ['pirads', 'psad_log', 'cspca']:
    cv_auc(make_model(C=1.0), X[[col]], y, f'  {col} alone')

print('\nFULL MODEL (all features, C=0.1)')
base_scores = cv_auc(make_model(C=0.1), X, y, '  logistic regression')

BASELINES (one feature at a time)


  pirads alone                           AUC = 0.618  (± 0.134)


  psad_log alone                         AUC = 0.435  (± 0.108)


  cspca alone                            AUC = 0.557  (± 0.134)

FULL MODEL (all features, C=0.1)


  logistic regression                    AUC = 0.686  (± 0.119)


**How to read this:** compare the full model against the single-feature baselines. The `±` is the spread over 50 splits; if two models differ by less than that spread, you cannot say one is truly better.

## 4. Tuning the regularization strength (nested CV)

**Why tune?** `C` decides how simple the model is. Too small → underfits; too large → memorises noise.

**Why nested?** If we pick the best `C` using the *same* scores we report, the result is slightly too optimistic (we chose the winner of a contest and then reported its prize).
- **Inner loop** (`GridSearchCV`): picks `C` using only the training part.
- **Outer loop**: scores that choice on patients the inner loop never saw.

We keep the grid small and use fewer outer repeats to keep it fast.

In [4]:
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=7)

tuned = GridSearchCV(make_model(), param_grid={'model__C': C_GRID},
                     scoring='roc_auc', cv=inner_cv)

nested = cross_validate(tuned, X, y, cv=outer_cv, scoring='roc_auc', return_estimator=True)
print(f"Nested-CV AUC = {nested['test_score'].mean():.3f}  (± {nested['test_score'].std():.3f})")
print('C chosen in each outer fold:', sorted(e.best_params_['model__C'] for e in nested['estimator']))

Nested-CV AUC = 0.708  (± 0.130)
C chosen in each outer fold: [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.003, 0.003, 0.003, 0.003, 0.003, 0.01, 0.01, 0.01, 0.01, 0.03, 0.03, 0.03, 0.1, 0.3, 1]


**How to read this:** the nested AUC is the number to report. If the chosen `C` values jump all over the grid, the data cannot clearly tell the best strength, another sign of how small the dataset is.

## 5. Experiment: do the MRI embeddings help?

**Why only as an experiment:** a 1024-number embedding for 89 patients is far more columns than rows, so the model could easily overfit. We squeeze it with **PCA** down to a few components. PCA is placed *inside* the pipeline, so it learns only from the training patients of each fold (no leakage). We keep this block separate and compare it to the tabular-only model with the same CV.

In [5]:
emb_cols = meta['embedding_columns']
X_emb_all = pd.concat([X_all.reset_index(drop=True), emb[emb_cols]], axis=1)
X_emb = X_emb_all[is_labelled.values]

def make_model_with_embeddings(n_components, C=0.1):
    prep = ColumnTransformer([
        ('tabular', Pipeline([('imp', SimpleImputer(strategy='median')),
                              ('sc',  StandardScaler())]), FEATURES),
        ('embed',   Pipeline([('imp', SimpleImputer(strategy='median')),   # 2 labelled patients have no embedding
                              ('sc',  StandardScaler()),
                              ('pca', PCA(n_components=n_components, random_state=0))]), emb_cols),
    ])
    return Pipeline([('prep', prep), ('model', LogisticRegression(C=C, max_iter=5000))])

print('tabular only                         AUC = %.3f' % base_scores.mean())
for k in [3, 5, 10]:
    cv_auc(make_model_with_embeddings(k), X_emb, y, f'tabular + {k} embedding PCs')

tabular only                         AUC = 0.686


tabular + 3 embedding PCs                AUC = 0.653  (± 0.121)


tabular + 5 embedding PCs                AUC = 0.631  (± 0.121)


tabular + 10 embedding PCs               AUC = 0.596  (± 0.140)


**How to read this:** keep embeddings only if the AUC is *clearly* above the tabular-only line (by more than the ± spread). Otherwise the simpler model wins.

## 6. Final model and its coefficients

**Why:** after we have trusted the evaluation, we fit one last model on **all 91 labelled patients** (more data = better model). `GridSearchCV` picks `C` once more, with CV on all patients.

**Reading the coefficients:** because features are standardised, each coefficient is "effect of a 1-standard-deviation increase". `odds ratio = exp(coef)`: above 1 raises the chance of *biopsy*, below 1 lowers it. Features are correlated, so treat this as a guide to the model, not as proof of causation.

In [6]:
final_search = GridSearchCV(make_model(), {'model__C': C_GRID}, scoring='roc_auc', cv=inner_cv).fit(X, y)
final_model  = final_search.best_estimator_
print('chosen C:', final_search.best_params_['model__C'])

coefs = pd.Series(final_model.named_steps['model'].coef_[0], index=FEATURES)
table = pd.DataFrame({'coef': coefs, 'odds_ratio_per_SD': np.exp(coefs)})
print('\nTop features pushing TOWARD biopsy:');   print(table.sort_values('coef', ascending=False).head(8).round(3))
print('\nTop features pushing AWAY from biopsy:'); print(table.sort_values('coef').head(8).round(3))

chosen C: 0.003

Top features pushing TOWARD biopsy:
                                 coef  odds_ratio_per_SD
pirads                          0.040              1.040
enc_type_Follow-up - PSA + MRI  0.028              1.028
bx_cat_None                     0.028              1.028
weight_kg                       0.018              1.019
cspca                           0.017              1.017
smoking_status_current          0.016              1.017
height_cm                       0.016              1.016
psa_first_log                   0.016              1.016

Top features pushing AWAY from biopsy:
                                               coef  odds_ratio_per_SD
prior_isup_grade                             -0.034              0.966
has_prior_isup                               -0.034              0.967
bx_cat_Positive                              -0.032              0.968
enc_type_Re-evaluation - prior PCa diagnosis -0.032              0.968
lesion_max_mm                          

## 7. Out-of-fold predictions: errors, threshold, calibration

**Why out-of-fold (OOF)?** For every patient we want a prediction from a model that **never saw that patient**. We get it with CV, repeating 10 times and averaging, so each patient has one stable, honest risk value. All checks below use these.

In [7]:
best_C = final_search.best_params_['model__C']
oof = np.zeros(len(y))
for seed in range(10):
    for train_idx, test_idx in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        m = make_model(best_C).fit(X.iloc[train_idx], y.iloc[train_idx])
        oof[test_idx] += m.predict_proba(X.iloc[test_idx])[:, 1] / 10

print('AUC of out-of-fold predictions: %.3f' % roc_auc_score(y, oof))

AUC of out-of-fold predictions: 0.723


### 7a. Choosing a threshold
**Why:** a biopsy decision is about trade-offs. Missing a cancer (false negative) is worse than an unnecessary biopsy (false positive), so we pick the threshold that keeps **sensitivity ≥ 90%** and look at what specificity remains. *Caveat:* the threshold is chosen on the same predictions we report, so treat it as an estimate.

In [8]:
fpr, tpr, thr = roc_curve(y, oof)
i = np.where(tpr >= 0.90)[0][0]
threshold = thr[i]
pred = (oof >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print(f'threshold = {threshold:.2f}')
print(f'sensitivity (biopsy cases found): {tp/(tp+fn):.2f}   specificity (unnecessary biopsies avoided): {tn/(tn+fp):.2f}')
print(pd.DataFrame([[tn, fp], [fn, tp]], index=['truth: no biopsy', 'truth: biopsy'],
                   columns=['pred: no biopsy', 'pred: biopsy']))

threshold = 0.57
sensitivity (biopsy cases found): 0.91   specificity (unnecessary biopsies avoided): 0.17
                  pred: no biopsy  pred: biopsy
truth: no biopsy                6            29
truth: biopsy                   5            51


### 7b. Calibration
**Why:** if the model says "70% risk", about 70 of 100 such patients should truly need a biopsy. We group patients into 4 risk bands and compare *predicted* vs *actual* rates. With 91 patients the bands are noisy; large gaps matter, small ones do not.

In [9]:
bands = pd.qcut(oof, 4, labels=['lowest', 'low-mid', 'high-mid', 'highest'])
print(pd.DataFrame({'predicted': oof, 'actual': y.values}).groupby(bands).mean().round(2).assign(n=pd.Series(bands).value_counts()))

          predicted  actual   n
lowest         0.56    0.39  23
low-mid        0.60    0.48  23
high-mid       0.64    0.77  22
highest        0.67    0.83  23


**Reading the table above:** if the *predicted* column is squeezed into a narrow range while the *actual* column spreads wider, the strong regularization has made the model **under-confident**. Its ranking of patients (AUC) can still be useful, but the probabilities should not be quoted literally. A common fix is recalibrating them (e.g. Platt scaling) inside CV.

### 7c. Using `reasoning_audit.csv` the legitimate way
**Why:** this file was written *after* the decision, so it must never be a feature. But it tells us how **confident the clinician was**. If our model's mistakes are concentrated in the clinician's "uncertain" cases, the errors are understandable (those cases are genuinely hard or the label is shaky).

In [10]:
audit = pd.read_csv(f'{DATA_DIR}/reasoning_audit.csv')[['case_id', 'confidence']]
check = pd.DataFrame({'case_id': ids.values, 'y': y.values, 'p': oof}).merge(audit, on='case_id', how='left')
check['abs_error'] = (check['y'] - check['p']).abs()
print(check.groupby('confidence').agg(patients=('y', 'size'), mean_abs_error=('abs_error', 'mean')).round(3))

            patients  mean_abs_error
confidence                          
borderline        18           0.476
clear             58           0.451
uncertain         15           0.471


**How to read this:** a higher mean error in the low-confidence group means the model struggles where the clinician also struggled. If errors are the same everywhere, confidence is not explaining them. Do not over-interpret the small groups.

## 8. Predict the unlabelled patients

**Why:** this is the purpose of the model. Remember these patients have **no ground truth**, so these numbers are risk estimates we cannot score. Use them for review, not as proof of accuracy.

In [11]:
risk = final_model.predict_proba(X_all[~is_labelled])[:, 1]
out = pd.DataFrame({'case_id': ids_all[~is_labelled].values,
                    'biopsy_risk': risk.round(3),
                    'recommend_biopsy': (risk >= threshold).astype(int)})
out.to_csv('phase2_unlabelled_predictions.csv', index=False)
print(out['recommend_biopsy'].value_counts().rename({1: 'biopsy', 0: 'no biopsy'}).to_string())
out.head()

recommend_biopsy
biopsy       81
no biopsy    23


,case_id,biopsy_risk,recommend_biopsy
0,PT-pseudo_598f1d8f4e08,0.615,1
1,PT-pseudo_5af00f9153c2,0.571,1
2,PT-pseudo_5dc2eb70fbd7,0.659,1
3,PT-pseudo_5f263f325019,0.593,1
4,PT-pseudo_62c93b0c3dba,0.635,1


**Note:** treat `biopsy_risk` as a *ranking score* (higher = more likely to need biopsy) rather than an exact probability, for the calibration reason in 7b.

## Summary checklist (what to report)
- Nested-CV AUC with its spread (section 4), next to the baselines (section 3)
- Whether embeddings helped (section 5)
- Top coefficients (section 6) and a sanity check with the clinician's view
- Threshold, sensitivity and specificity (7a), calibration (7b)
- Honest caveat: 91 labelled patients, so the uncertainty is large